# Notebook
#### The goal of this notebook is to combine annotation files and other files into 1 main data file that we can use for downstream evaluations

In [2]:
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [15]:
from rdma.utils.data import read_json_file, print_json_structure
data_path = "data/medical_students_data/existing_annotations"
# jeffrey = read_json_file(data_path + '/jennifer_rd_step4_eval.json')
jeffrey = read_json_file(data_path + '/jeffrey_rd_step4_eval.json')
# testing what happens
jennifer = read_json_file(data_path + '/jennifer_rd_step4_eval.json')
lauren = read_json_file(data_path + '/lauren_rd_step4_eval.json')
mya = read_json_file(data_path + '/mya_rd_step4_eval.json')

jeffrey_p2 = read_json_file(data_path + '/jeffrey_p2.json')
jennifer_p2 = read_json_file(data_path + '/jennifer_p2.json')
lauren_p2 = read_json_file(data_path + '/lauren_p2.json')
mya_p2 = read_json_file(data_path + '/mya_p2.json')

def retrieve_rare_diseases(annotations):
    patient_diseases = {} # patient id : { disease codes, disease names, pairings }
    for annotation in annotations['corrected_annotations']:
        if annotation['is_rare_disease']:
            patient_id = annotation['document_id']
            disease_code = annotation['orpha_code']
            disease_name = annotation['entity']
            if patient_id not in patient_diseases:
                patient_diseases[patient_id] = {
                    'disease_codes': set(),
                    'disease_names': set(),
                    'pairings': set()
                }
            if annotation['is_rare_disease']:
                patient_diseases[patient_id]['disease_codes'].add(disease_code)
                patient_diseases[patient_id]['disease_names'].add(disease_name)
                patient_diseases[patient_id]['pairings'].add((disease_code, disease_name))
    return patient_diseases

jeffrey_rd = retrieve_rare_diseases(jeffrey)
jeffrey_rd_p2 = retrieve_rare_diseases(jeffrey_p2)
jennifer_rd = retrieve_rare_diseases(jennifer)
jennifer_rd_p2 = retrieve_rare_diseases(jennifer_p2)
lauren_rd = retrieve_rare_diseases(lauren)
lauren_rd_p2 = retrieve_rare_diseases(lauren_p2)
mya_rd = retrieve_rare_diseases(mya)
mya_rd_p2 = retrieve_rare_diseases(mya_p2)




                


In [12]:
from collections import defaultdict
from rdma.utils.data import save_json_structure

def merge_annotator_parts(part1_data, part2_data):
    """
    Merge two parts of an annotator's data by combining their corrected_annotations.
    
    Args:
        part1_data: Dictionary containing part 1 annotations
        part2_data: Dictionary containing part 2 annotations
    
    Returns:
        Dictionary with merged corrected_annotations
    """
    merged_annotations = []
    
    # Combine annotations from both parts
    if 'corrected_annotations' in part1_data:
        merged_annotations.extend(part1_data['corrected_annotations'])
    
    if 'corrected_annotations' in part2_data:
        merged_annotations.extend(part2_data['corrected_annotations'])
    
    return {'corrected_annotations': merged_annotations}

def extract_rare_diseases_by_patient(merged_annotations):
    """
    Extract rare diseases organized by patient_id from merged annotations.
    
    Args:
        merged_annotations: Dictionary with 'corrected_annotations' key
    
    Returns:
        Dictionary: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
    """
    patient_diseases = defaultdict(lambda: {
        'orpha_codes': [],
        'disease_entities': [], 
        'pairings': []
    })
    
    for annotation in merged_annotations['corrected_annotations']:
        if annotation.get('is_rare_disease', False):
            patient_id = annotation.get('document_id')
            disease_code = annotation.get('orpha_code')
            disease_entity = annotation.get('entity')
            
            if patient_id and disease_code and disease_entity:
                # Avoid duplicates within the same annotator
                if disease_code not in patient_diseases[patient_id]['orpha_codes']:
                    patient_diseases[patient_id]['orpha_codes'].append(disease_code)
                
                if disease_entity not in patient_diseases[patient_id]['disease_entities']:
                    patient_diseases[patient_id]['disease_entities'].append(disease_entity)
                
                pairing = (disease_code, disease_entity)
                if pairing not in patient_diseases[patient_id]['pairings']:
                    patient_diseases[patient_id]['pairings'].append(pairing)
    
    return dict(patient_diseases)

def find_high_agreement_rare_diseases(annotator_parts_dict, agreement_threshold=3):
    """
    Find high agreement rare diseases across all annotators.
    
    Args:
        annotator_parts_dict: Dictionary with structure:
            {
                'annotator_name': {
                    'part1': part1_data,
                    'part2': part2_data
                }
            }
        agreement_threshold: Minimum number of annotators that must agree (default: 3)
    
    Returns:
        Dictionary: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
        Contains only high agreement cases.
    """
    
    # Step 1: Merge parts for each annotator and extract rare diseases
    annotator_rare_diseases = {}
    
    for annotator_name, parts in annotator_parts_dict.items():
        print(f"Processing annotator: {annotator_name}")
        
        # Merge the two parts
        merged_data = merge_annotator_parts(parts['part1'], parts['part2'])
        
        # Extract rare diseases by patient
        patient_diseases = extract_rare_diseases_by_patient(merged_data)
        annotator_rare_diseases[annotator_name] = patient_diseases
        
        print(f"  - Found rare diseases in {len(patient_diseases)} patients")
    
    # Step 2: Track agreement across annotators
    # For orpha codes by patient
    patient_code_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> orpha_code -> set of annotators
    # For disease entities by patient  
    patient_entity_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> disease_entity -> set of annotators
    # For pairings by patient
    patient_pairing_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> (code,entity) -> set of annotators
    
    # Collect votes from all annotators
    for annotator_name, patient_diseases in annotator_rare_diseases.items():
        for patient_id, disease_info in patient_diseases.items():
            
            # Vote for each orpha code
            for orpha_code in disease_info['orpha_codes']:
                patient_code_votes[patient_id][orpha_code].add(annotator_name)
            
            # Vote for each disease entity
            for disease_entity in disease_info['disease_entities']:
                patient_entity_votes[patient_id][disease_entity].add(annotator_name)
            
            # Vote for each pairing
            for pairing in disease_info['pairings']:
                patient_pairing_votes[patient_id][pairing].add(annotator_name)
    
    # Step 3: Filter for high agreement cases
    high_agreement_results = defaultdict(lambda: {
        'orpha_codes': [],
        'disease_entities': [],
        'pairings': []
    })
    
    # Find high agreement orpha codes
    for patient_id, code_votes in patient_code_votes.items():
        for orpha_code, annotators in code_votes.items():
            if len(annotators) > agreement_threshold:
                high_agreement_results[patient_id]['orpha_codes'].append(orpha_code)
    
    # Find high agreement disease entities
    for patient_id, entity_votes in patient_entity_votes.items():
        for disease_entity, annotators in entity_votes.items():
            if len(annotators) > agreement_threshold:
                high_agreement_results[patient_id]['disease_entities'].append(disease_entity)
    
    # Find high agreement pairings
    for patient_id, pairing_votes in patient_pairing_votes.items():
        for pairing, annotators in pairing_votes.items():
            if len(annotators) > agreement_threshold:
                high_agreement_results[patient_id]['pairings'].append(pairing)
    
    # Convert defaultdict to regular dict and remove empty patients
    final_results = {}
    for patient_id, disease_info in high_agreement_results.items():
        if (disease_info['orpha_codes'] or 
            disease_info['disease_entities'] or 
            disease_info['pairings']):
            final_results[patient_id] = disease_info
    
    return dict(final_results)

def print_high_agreement_summary(high_agreement_data):
    """Print a summary of high agreement results."""
    
    print("=" * 80)
    print("HIGH AGREEMENT RARE DISEASES BY PATIENT")
    print("=" * 80)
    
    if not high_agreement_data:
        print("No high agreement cases found.")
        return
    
    print(f"Found high agreement rare diseases in {len(high_agreement_data)} patients\n")
    
    for patient_id, disease_info in high_agreement_data.items():
        print(f"Patient ID: {patient_id}")
        print("-" * 40)
        
        if disease_info['orpha_codes']:
            print(f"  High Agreement Orpha Codes ({len(disease_info['orpha_codes'])}): {disease_info['orpha_codes']}")
        
        if disease_info['disease_entities']:
            print(f"  High Agreement Disease Entities ({len(disease_info['disease_entities'])}): {disease_info['disease_entities']}")
        
        if disease_info['pairings']:
            print(f"  High Agreement Pairings ({len(disease_info['pairings'])}):")
            for code, entity in disease_info['pairings']:
                print(f"    - ({code}, {entity})")
        
        print()

# Usage example:
# Structure your data like this:
annotator_data = {
    'jeffrey': {
        'part1': jeffrey,  # your jeffrey variable
        'part2': jeffrey_p2  # your jeffrey_p2 variable
    },
    'jennifer': {
        'part1': jennifer,
        'part2': jennifer_p2
    },
    'lauren': {
        'part1': lauren,
        'part2': lauren_p2
    },
    'mya': {
        'part1': mya,
        'part2': mya_p2
    }
}

# Find high agreement cases (>3 annotators agree)
high_agreement_rare_diseases = find_high_agreement_rare_diseases(annotator_data, agreement_threshold=3)

# Print summary
print_high_agreement_summary(high_agreement_rare_diseases)

# Access the results:
# high_agreement_rare_diseases is now a dictionary where:
# - Keys are patient_ids 
# - Values are dictionaries with 'orpha_codes', 'disease_entities', and 'pairings' lists
# containing only the high agreement cases

print(f"\nExample access:")
print(f"Number of patients with high agreement: {len(high_agreement_rare_diseases)}")
for patient_id in list(high_agreement_rare_diseases.keys())[:3]:  # Show first 3 patients
    print(f"Patient {patient_id} high agreement items:")
    print(f"  - Codes: {high_agreement_rare_diseases[patient_id]['orpha_codes']}")
    print(f"  - Entities: {high_agreement_rare_diseases[patient_id]['disease_entities']}")
    print(f"  - Pairings: {high_agreement_rare_diseases[patient_id]['pairings']}")

# Save functions
import json
import pickle
from datetime import datetime

def save_high_agreement_results(high_agreement_data, base_filename=None, save_json=True, save_pickle=True):
    """
    Save high agreement results to file(s).
    
    Args:
        high_agreement_data: Dictionary to save
        base_filename: Base filename (without extension). If None, uses timestamp
        save_json: Whether to save as JSON (default: True)
        save_pickle: Whether to save as pickle (default: True)
    
    Returns:
        Dictionary with saved file paths
    """
    if base_filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"high_agreement_rare_diseases_{timestamp}"
    
    saved_files = {}
    
    # Save as JSON
    if save_json:
        json_filename = f"{base_filename}.json"
        try:
            # Convert tuples to lists for JSON serialization
            json_data = {}
            for patient_id, disease_info in high_agreement_data.items():
                json_data[patient_id] = {
                    'orpha_codes': disease_info['orpha_codes'],
                    'disease_entities': disease_info['disease_entities'],
                    'pairings': [list(pairing) for pairing in disease_info['pairings']]  # Convert tuples to lists
                }
            
            with open(json_filename, 'w', encoding='utf-8') as f:
                json.dump(json_data, f, indent=2, ensure_ascii=False)
            
            saved_files['json'] = json_filename
            print(f"Saved JSON to: {json_filename}")
        except Exception as e:
            print(f"Error saving JSON: {e}")
    
    # Save as pickle (preserves exact data types including tuples)
    if save_pickle:
        pickle_filename = f"{base_filename}.pkl"
        try:
            with open(pickle_filename, 'wb') as f:
                pickle.dump(high_agreement_data, f)
            
            saved_files['pickle'] = pickle_filename
            print(f"Saved pickle to: {pickle_filename}")
        except Exception as e:
            print(f"Error saving pickle: {e}")
    
    return saved_files

def load_high_agreement_results(filename):
    """
    Load high agreement results from file.
    
    Args:
        filename: Path to the file (.json or .pkl)
    
    Returns:
        Dictionary with high agreement data
    """
    if filename.endswith('.json'):
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Convert lists back to tuples for pairings
        for patient_id, disease_info in data.items():
            data[patient_id]['pairings'] = [tuple(pairing) for pairing in disease_info['pairings']]
        
        return data
    
    elif filename.endswith('.pkl'):
        with open(filename, 'rb') as f:
            return pickle.load(f)
    
    else:
        raise ValueError("File must be .json or .pkl")

def save_summary_report(high_agreement_data, filename=None):
    """
    Save a human-readable summary report.
    
    Args:
        high_agreement_data: Dictionary to summarize
        filename: Output filename. If None, uses timestamp
    
    Returns:
        String with filename
    """
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"high_agreement_summary_{timestamp}.txt"
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("HIGH AGREEMENT RARE DISEASES BY PATIENT\n")
        f.write("=" * 80 + "\n\n")
        
        if not high_agreement_data:
            f.write("No high agreement cases found.\n")
            return filename
        
        f.write(f"Found high agreement rare diseases in {len(high_agreement_data)} patients\n\n")
        
        for patient_id, disease_info in high_agreement_data.items():
            f.write(f"Patient ID: {patient_id}\n")
            f.write("-" * 40 + "\n")
            
            if disease_info['orpha_codes']:
                f.write(f"  High Agreement Orpha Codes ({len(disease_info['orpha_codes'])}): {disease_info['orpha_codes']}\n")
            
            if disease_info['disease_entities']:
                f.write(f"  High Agreement Disease Entities ({len(disease_info['disease_entities'])}): {disease_info['disease_entities']}\n")
            
            if disease_info['pairings']:
                f.write(f"  High Agreement Pairings ({len(disease_info['pairings'])}):\n")
                for code, entity in disease_info['pairings']:
                    f.write(f"    - ({code}, {entity})\n")
            
            f.write("\n")
    
    print(f"Saved summary report to: {filename}")
    return filename


def add_phenotypes_to_high_agreement(high_agreement_data, phenotypes_per_patient):
    """
    Add matched phenotypes to high agreement rare diseases data.
    
    Args:
        high_agreement_data: Dictionary from find_high_agreement_rare_diseases()
                            Structure: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
        phenotypes_per_patient: Dictionary with patient_id -> phenotype data
                               Structure: patient_id -> {matched_phenotypes: list, original_text: str, stats: dict}
    
    Returns:
        Updated high_agreement_data with phenotypes added
        New structure: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list, matched_phenotypes: list}
    """
    updated_data = high_agreement_data.copy()
    
    total_patients = len(updated_data)
    patients_with_phenotypes = 0
    total_phenotypes_added = 0
    
    print(f"Adding phenotypes to {total_patients} patients with high agreement rare diseases...")
    print("-" * 60)
    
    for patient_id in updated_data.keys():
        if patient_id in phenotypes_per_patient:
            # Only add the matched_phenotypes list, excluding original_text and stats
            phenotypes_list = phenotypes_per_patient[patient_id]['matched_phenotypes']
            updated_data[patient_id]['matched_phenotypes'] = phenotypes_list
            
            patients_with_phenotypes += 1
            total_phenotypes_added += len(phenotypes_list)
            
            print(f"Patient {patient_id}: Added {len(phenotypes_list)} phenotypes")
        else:
            # Add empty list if no phenotypes found
            updated_data[patient_id]['matched_phenotypes'] = []
            print(f"Patient {patient_id}: No phenotypes found - added empty list")
    
    print("-" * 60)
    print(f"Summary:")
    print(f"  - Total patients processed: {total_patients}")
    print(f"  - Patients with phenotypes: {patients_with_phenotypes}")
    print(f"  - Patients without phenotypes: {total_patients - patients_with_phenotypes}")
    print(f"  - Total phenotypes added: {total_phenotypes_added}")
    print(f"  - Average phenotypes per patient with data: {total_phenotypes_added/patients_with_phenotypes if patients_with_phenotypes > 0 else 0:.1f}")
    
    return updated_data

# Usage example:
# combined_data = add_phenotypes_to_high_agreement(high_agreement_rare_diseases, phenotypes_per_patient)
# print_json_structure(phenotypes_per_patient, indent=2)

# Example usage after getting your results:
# high_agreement_rare_diseases = find_high_agreement_rare_diseases(annotator_data, agreement_threshold=3)

# SIMPLE JSON SAVE - Just use this:
# save_high_agreement_results(high_agreement_rare_diseases, "consensus_results", save_json=True, save_pickle=False)

# Or even simpler - save only JSON with timestamp:
# save_high_agreement_results(high_agreement_rare_diseases, save_pickle=False)

# Complete workflow example:
# 1. Get high agreement rare diseases
high_agreement_rare_diseases = find_high_agreement_rare_diseases(annotator_data, agreement_threshold=3)

# 2. Load phenotypes data
phenotypes_data = read_json_file("data/medical_students_data/LLM_passes/step3_match_hpo_1000_context_output.json")
phenotypes_per_patient = phenotypes_data['results']

# 3. Combine phenotypes with high agreement data
combined_data = add_phenotypes_to_high_agreement(high_agreement_rare_diseases, phenotypes_per_patient)
save_json_structure(combined_data, "data/medical_students_data/high_agreement_with_phenotypes.json")
# 4. Save combined data to JSON
# save_high_agreement_results(combined_data, "data/medical_students_data/high_agreement_with_phenotypes", save_json=True, save_pickle=False)

# Load the JSON later:
# loaded_data = load_high_agreement_results("high_agreement_with_phenotypes.json")

Processing annotator: jeffrey
  - Found rare diseases in 184 patients
Processing annotator: jennifer
  - Found rare diseases in 181 patients
Processing annotator: lauren
  - Found rare diseases in 170 patients
Processing annotator: mya
  - Found rare diseases in 183 patients
HIGH AGREEMENT RARE DISEASES BY PATIENT
Found high agreement rare diseases in 160 patients

Patient ID: 10402135
----------------------------------------
  High Agreement Orpha Codes (1): ['2942']
  High Agreement Disease Entities (1): ['postpoliomyelitis syndrome']
  High Agreement Pairings (1):
    - (2942, postpoliomyelitis syndrome)

Patient ID: 10877494
----------------------------------------
  High Agreement Orpha Codes (3): ['86886', 'Orpha:98375', 'ORPHA:223735']
  High Agreement Disease Entities (3): ['angioimmunoblastic t-cell lymphoma', 'autoimmune hemolytic anemia', 'lymphoma']
  High Agreement Pairings (3):
    - (86886, angioimmunoblastic t-cell lymphoma)
    - (Orpha:98375, autoimmune hemolytic anem

In [16]:
from collections import defaultdict
import numpy as np
from sklearn.metrics import cohen_kappa_score
from itertools import combinations
import pandas as pd
from rdma.utils.data import save_json_structure

def merge_annotator_parts(part1_data, part2_data):
    """
    Merge two parts of an annotator's data by combining their corrected_annotations.
    
    Args:
        part1_data: Dictionary containing part 1 annotations
        part2_data: Dictionary containing part 2 annotations
    
    Returns:
        Dictionary with merged corrected_annotations
    """
    merged_annotations = []
    
    # Combine annotations from both parts
    if 'corrected_annotations' in part1_data:
        merged_annotations.extend(part1_data['corrected_annotations'])
    
    if 'corrected_annotations' in part2_data:
        merged_annotations.extend(part2_data['corrected_annotations'])
    
    return {'corrected_annotations': merged_annotations}

def extract_rare_diseases_by_patient(merged_annotations):
    """
    Extract rare diseases organized by patient_id from merged annotations.
    
    Args:
        merged_annotations: Dictionary with 'corrected_annotations' key
    
    Returns:
        Dictionary: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
    """
    patient_diseases = defaultdict(lambda: {
        'orpha_codes': [],
        'disease_entities': [], 
        'pairings': []
    })
    
    for annotation in merged_annotations['corrected_annotations']:
        if annotation.get('is_rare_disease', False):
            patient_id = annotation.get('document_id')
            disease_code = annotation.get('orpha_code')
            disease_entity = annotation.get('entity')
            
            if patient_id and disease_code and disease_entity:
                # Avoid duplicates within the same annotator
                if disease_code not in patient_diseases[patient_id]['orpha_codes']:
                    patient_diseases[patient_id]['orpha_codes'].append(disease_code)
                
                if disease_entity not in patient_diseases[patient_id]['disease_entities']:
                    patient_diseases[patient_id]['disease_entities'].append(disease_entity)
                
                pairing = (disease_code, disease_entity)
                if pairing not in patient_diseases[patient_id]['pairings']:
                    patient_diseases[patient_id]['pairings'].append(pairing)
    
    return dict(patient_diseases)

def get_all_patients_from_annotations(annotator_parts_dict):
    """Get all unique patient IDs across all annotators."""
    all_patients = set()
    
    for annotator_name, parts in annotator_parts_dict.items():
        merged_data = merge_annotator_parts(parts['part1'], parts['part2'])
        
        for annotation in merged_data['corrected_annotations']:
            patient_id = annotation.get('document_id')
            if patient_id:
                all_patients.add(patient_id)
    
    return sorted(all_patients)

def compute_irr_metrics(annotator_parts_dict):
    """
    Compute inter-rater reliability metrics for rare disease annotations.
    
    Args:
        annotator_parts_dict: Dictionary with structure:
            {
                'annotator_name': {
                    'part1': part1_data,
                    'part2': part2_data
                }
            }
    
    Returns:
        Dictionary containing IRR metrics and statistics
    """
    print("Computing Inter-Rater Reliability Metrics...")
    print("=" * 60)
    
    # Step 1: Get all annotators and patients
    annotators = list(annotator_parts_dict.keys())
    all_patients = get_all_patients_from_annotations(annotator_parts_dict)
    
    print(f"Annotators: {annotators}")
    print(f"Total patients: {len(all_patients)}")
    
    # Step 2: Create binary matrix for rare disease presence (patient-level)
    # Matrix: rows = patients, columns = annotators
    # Value = 1 if annotator found any rare disease for this patient, 0 otherwise
    patient_level_matrix = np.zeros((len(all_patients), len(annotators)), dtype=int)
    
    # Step 3: Process each annotator
    annotator_rare_diseases = {}
    annotator_stats = {}
    
    for ann_idx, annotator_name in enumerate(annotators):
        parts = annotator_parts_dict[annotator_name]
        merged_data = merge_annotator_parts(parts['part1'], parts['part2'])
        patient_diseases = extract_rare_diseases_by_patient(merged_data)
        annotator_rare_diseases[annotator_name] = patient_diseases
        
        # Fill in the matrix
        patients_with_rare_diseases = 0
        total_rare_disease_annotations = 0
        
        for pat_idx, patient_id in enumerate(all_patients):
            if patient_id in patient_diseases and (
                patient_diseases[patient_id]['orpha_codes'] or
                patient_diseases[patient_id]['disease_entities'] or
                patient_diseases[patient_id]['pairings']
            ):
                patient_level_matrix[pat_idx, ann_idx] = 1
                patients_with_rare_diseases += 1
                total_rare_disease_annotations += len(patient_diseases[patient_id]['pairings'])
        
        annotator_stats[annotator_name] = {
            'patients_with_rare_diseases': patients_with_rare_diseases,
            'total_patients': len(all_patients),
            'rare_disease_prevalence': patients_with_rare_diseases / len(all_patients) * 100,
            'total_rare_disease_annotations': total_rare_disease_annotations
        }
        
        print(f"{annotator_name}:")
        print(f"  - Patients with rare diseases: {patients_with_rare_diseases}/{len(all_patients)} ({patients_with_rare_diseases/len(all_patients)*100:.1f}%)")
        print(f"  - Total rare disease annotations: {total_rare_disease_annotations}")
    
    print()
    
    # Step 4: Compute pairwise Cohen's Kappa
    kappa_results = {}
    kappa_values = []
    
    print("Pairwise Cohen's Kappa (Patient-level rare disease presence):")
    print("-" * 50)
    
    for ann1, ann2 in combinations(annotators, 2):
        ann1_idx = annotators.index(ann1)
        ann2_idx = annotators.index(ann2)
        
        y1 = patient_level_matrix[:, ann1_idx]
        y2 = patient_level_matrix[:, ann2_idx]
        
        kappa = cohen_kappa_score(y1, y2)
        kappa_results[f"{ann1}_vs_{ann2}"] = kappa
        kappa_values.append(kappa)
        
        print(f"{ann1} vs {ann2}: κ = {kappa:.3f}")
    
    mean_kappa = np.mean(kappa_values)
    print(f"\nMean pairwise κ = {mean_kappa:.3f}")
    
    # Step 5: Compute overall agreement statistics
    # Percentage agreement
    agreement_counts = np.sum(patient_level_matrix, axis=1)  # How many annotators agreed on each patient
    
    perfect_agreement = np.sum(agreement_counts == len(annotators))  # All annotators agree
    no_agreement = np.sum(agreement_counts == 0)  # No annotator found rare disease
    partial_agreement = len(all_patients) - perfect_agreement - no_agreement
    
    # Majority agreement (>50% of annotators)
    majority_threshold = len(annotators) / 2
    majority_agreement = np.sum(agreement_counts > majority_threshold)
    
    print(f"\nOverall Agreement Statistics:")
    print(f"- Perfect agreement (all annotators): {perfect_agreement}/{len(all_patients)} ({perfect_agreement/len(all_patients)*100:.1f}%)")
    print(f"- No rare diseases found by any annotator: {no_agreement}/{len(all_patients)} ({no_agreement/len(all_patients)*100:.1f}%)")
    print(f"- Partial agreement: {partial_agreement}/{len(all_patients)} ({partial_agreement/len(all_patients)*100:.1f}%)")
    print(f"- Majority agreement (>{majority_threshold:.0f} annotators): {majority_agreement}/{len(all_patients)} ({majority_agreement/len(all_patients)*100:.1f}%)")
    
    # Step 6: Compute Fleiss' Kappa for multiple raters
    def compute_fleiss_kappa(matrix):
        """Compute Fleiss' Kappa for multiple raters."""
        n_items, n_raters = matrix.shape
        n_categories = 2  # binary: rare disease present or not
        
        # Count agreements
        p_categories = np.zeros(n_categories)
        
        for item_idx in range(n_items):
            ratings = matrix[item_idx, :]
            
            # Count category frequencies for this item
            count_0 = np.sum(ratings == 0)
            count_1 = np.sum(ratings == 1)
            
            p_categories[0] += count_0
            p_categories[1] += count_1
        
        # Overall proportion in each category
        p_categories = p_categories / (n_items * n_raters)
        
        # Compute observed agreement
        P_observed = 0
        for item_idx in range(n_items):
            ratings = matrix[item_idx, :]
            count_0 = np.sum(ratings == 0)
            count_1 = np.sum(ratings == 1)
            
            # Agreement for this item
            item_agreement = (count_0 * (count_0 - 1) + count_1 * (count_1 - 1)) / (n_raters * (n_raters - 1))
            P_observed += item_agreement
        
        P_observed = P_observed / n_items
        
        # Expected agreement
        P_expected = np.sum(p_categories ** 2)
        
        # Fleiss' Kappa
        if P_expected == 1:
            return 1.0  # Perfect agreement case
        
        fleiss_kappa = (P_observed - P_expected) / (1 - P_expected)
        return fleiss_kappa
    
    fleiss_kappa = compute_fleiss_kappa(patient_level_matrix)
    print(f"\nFleiss' Kappa (multi-rater): κ = {fleiss_kappa:.3f}")
    
    # Step 7: Analyze specific disease agreement
    # Get all unique diseases across all annotators
    all_orpha_codes = set()
    all_disease_entities = set()
    all_pairings = set()
    
    for annotator_diseases in annotator_rare_diseases.values():
        for patient_diseases in annotator_diseases.values():
            all_orpha_codes.update(patient_diseases['orpha_codes'])
            all_disease_entities.update(patient_diseases['disease_entities'])
            all_pairings.update(patient_diseases['pairings'])
    
    print(f"\nDisease Catalog Statistics:")
    print(f"- Unique Orpha codes across all annotators: {len(all_orpha_codes)}")
    print(f"- Unique disease entities across all annotators: {len(all_disease_entities)}")
    print(f"- Unique (code, entity) pairings: {len(all_pairings)}")
    
    # Return comprehensive results
    irr_results = {
        'patient_level_metrics': {
            'total_patients': len(all_patients),
            'annotators': annotators,
            'patient_level_matrix': patient_level_matrix.tolist(),
            'patient_ids': all_patients
        },
        'annotator_statistics': annotator_stats,
        'agreement_metrics': {
            'pairwise_kappa': kappa_results,
            'mean_pairwise_kappa': mean_kappa,
            'fleiss_kappa': fleiss_kappa
        },
        'agreement_counts': {
            'perfect_agreement': perfect_agreement,
            'no_agreement': no_agreement,
            'partial_agreement': partial_agreement,
            'majority_agreement': majority_agreement,
            'perfect_agreement_pct': perfect_agreement/len(all_patients)*100,
            'no_agreement_pct': no_agreement/len(all_patients)*100,
            'partial_agreement_pct': partial_agreement/len(all_patients)*100,
            'majority_agreement_pct': majority_agreement/len(all_patients)*100
        },
        'disease_catalog': {
            'unique_orpha_codes': len(all_orpha_codes),
            'unique_disease_entities': len(all_disease_entities),
            'unique_pairings': len(all_pairings),
            'all_orpha_codes': sorted(list(all_orpha_codes)),
            'all_disease_entities': sorted(list(all_disease_entities)),
            'all_pairings': sorted(list(all_pairings))
        }
    }
    
    return irr_results

def interpret_kappa(kappa_value):
    """Interpret Cohen's/Fleiss' Kappa values according to Landis & Koch (1977)."""
    if kappa_value < 0:
        return "Poor (less than chance agreement)"
    elif kappa_value < 0.20:
        return "Slight"
    elif kappa_value < 0.40:
        return "Fair"
    elif kappa_value < 0.60:
        return "Moderate"
    elif kappa_value < 0.80:
        return "Substantial"
    else:
        return "Almost Perfect"

def find_high_agreement_rare_diseases(annotator_parts_dict, agreement_threshold=3):
    """
    Find high agreement rare diseases across all annotators.
    
    Args:
        annotator_parts_dict: Dictionary with structure:
            {
                'annotator_name': {
                    'part1': part1_data,
                    'part2': part2_data
                }
            }
        agreement_threshold: Minimum number of annotators that must agree (default: 3)
    
    Returns:
        Dictionary: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
        Contains only high agreement cases.
    """
    
    # Step 1: Merge parts for each annotator and extract rare diseases
    annotator_rare_diseases = {}
    
    for annotator_name, parts in annotator_parts_dict.items():
        print(f"Processing annotator: {annotator_name}")
        
        # Merge the two parts
        merged_data = merge_annotator_parts(parts['part1'], parts['part2'])
        
        # Extract rare diseases by patient
        patient_diseases = extract_rare_diseases_by_patient(merged_data)
        annotator_rare_diseases[annotator_name] = patient_diseases
        
        print(f"  - Found rare diseases in {len(patient_diseases)} patients")
    
    # Step 2: Track agreement across annotators
    # For orpha codes by patient
    patient_code_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> orpha_code -> set of annotators
    # For disease entities by patient  
    patient_entity_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> disease_entity -> set of annotators
    # For pairings by patient
    patient_pairing_votes = defaultdict(lambda: defaultdict(set))  # patient_id -> (code,entity) -> set of annotators
    
    # Collect votes from all annotators
    for annotator_name, patient_diseases in annotator_rare_diseases.items():
        for patient_id, disease_info in patient_diseases.items():
            
            # Vote for each orpha code
            for orpha_code in disease_info['orpha_codes']:
                patient_code_votes[patient_id][orpha_code].add(annotator_name)
            
            # Vote for each disease entity
            for disease_entity in disease_info['disease_entities']:
                patient_entity_votes[patient_id][disease_entity].add(annotator_name)
            
            # Vote for each pairing
            for pairing in disease_info['pairings']:
                patient_pairing_votes[patient_id][pairing].add(annotator_name)
    
    # Step 3: Filter for high agreement cases
    high_agreement_results = defaultdict(lambda: {
        'orpha_codes': [],
        'disease_entities': [],
        'pairings': []
    })
    
    # Find high agreement orpha codes
    for patient_id, code_votes in patient_code_votes.items():
        for orpha_code, annotators in code_votes.items():
            if len(annotators) >= agreement_threshold:
                high_agreement_results[patient_id]['orpha_codes'].append(orpha_code)
    
    # Find high agreement disease entities
    for patient_id, entity_votes in patient_entity_votes.items():
        for disease_entity, annotators in entity_votes.items():
            if len(annotators) >= agreement_threshold:
                high_agreement_results[patient_id]['disease_entities'].append(disease_entity)
    
    # Find high agreement pairings
    for patient_id, pairing_votes in patient_pairing_votes.items():
        for pairing, annotators in pairing_votes.items():
            if len(annotators) >= agreement_threshold:
                high_agreement_results[patient_id]['pairings'].append(pairing)
    
    # Convert defaultdict to regular dict and remove empty patients
    final_results = {}
    for patient_id, disease_info in high_agreement_results.items():
        if (disease_info['orpha_codes'] or 
            disease_info['disease_entities'] or 
            disease_info['pairings']):
            final_results[patient_id] = disease_info
    
    return dict(final_results)

def print_comprehensive_analysis(annotator_parts_dict, agreement_threshold=3):
    """Print comprehensive analysis including IRR and high agreement results."""
    
    print("🔬 COMPREHENSIVE RARE DISEASE ANNOTATION ANALYSIS")
    print("=" * 80)
    
    # Compute IRR metrics
    irr_results = compute_irr_metrics(annotator_parts_dict)
    
    print("\n📊 INTERPRETATION OF AGREEMENT METRICS:")
    print("-" * 50)
    print(f"Mean Pairwise Cohen's κ: {irr_results['agreement_metrics']['mean_pairwise_kappa']:.3f} ({interpret_kappa(irr_results['agreement_metrics']['mean_pairwise_kappa'])})")
    print(f"Fleiss' κ (Multi-rater): {irr_results['agreement_metrics']['fleiss_kappa']:.3f} ({interpret_kappa(irr_results['agreement_metrics']['fleiss_kappa'])})")
    
    print(f"\n📈 RARE DISEASE PREVALENCE BY ANNOTATOR:")
    print("-" * 50)
    prevalences = []
    for annotator, stats in irr_results['annotator_statistics'].items():
        prevalence = stats['rare_disease_prevalence']
        prevalences.append(prevalence)
        print(f"{annotator}: {prevalence:.1f}% ({stats['patients_with_rare_diseases']}/{stats['total_patients']} patients)")
    
    mean_prevalence = np.mean(prevalences)
    std_prevalence = np.std(prevalences)
    print(f"\nMean prevalence: {mean_prevalence:.1f}% ± {std_prevalence:.1f}%")
    
    # Find high agreement cases
    print(f"\n🎯 HIGH AGREEMENT ANALYSIS (≥{agreement_threshold} annotators):")
    print("-" * 50)
    
    high_agreement_rare_diseases = find_high_agreement_rare_diseases(annotator_parts_dict, agreement_threshold)
    
    if high_agreement_rare_diseases:
        print(f"High agreement rare diseases found in {len(high_agreement_rare_diseases)} patients")
        
        total_consensus_codes = sum(len(data['orpha_codes']) for data in high_agreement_rare_diseases.values())
        total_consensus_entities = sum(len(data['disease_entities']) for data in high_agreement_rare_diseases.values())
        total_consensus_pairings = sum(len(data['pairings']) for data in high_agreement_rare_diseases.values())
        
        print(f"- Consensus Orpha codes: {total_consensus_codes}")
        print(f"- Consensus disease entities: {total_consensus_entities}")  
        print(f"- Consensus pairings: {total_consensus_pairings}")
        
        consensus_prevalence = len(high_agreement_rare_diseases) / irr_results['patient_level_metrics']['total_patients'] * 100
        print(f"- Consensus rare disease prevalence: {consensus_prevalence:.1f}%")
    else:
        print("No high agreement cases found.")
    
    return irr_results, high_agreement_rare_diseases

def convert_numpy_types(obj):
    """Recursively convert numpy types to native Python types for JSON serialization."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_numpy_types(item) for item in obj)
    else:
        return obj

def save_comprehensive_results(irr_results, high_agreement_data, base_filename=None):
    """Save both IRR results and high agreement data."""
    from datetime import datetime
    import json
    
    if base_filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"rare_disease_analysis_{timestamp}"
    
    # Convert numpy types to native Python types
    irr_results_clean = convert_numpy_types(irr_results)
    
    # Save IRR results
    irr_filename = f"{base_filename}_irr_metrics.json"
    with open(irr_filename, 'w') as f:
        json.dump(irr_results_clean, f, indent=2, ensure_ascii=False)
    print(f"Saved IRR metrics to: {irr_filename}")
    
    # Save high agreement results
    consensus_filename = f"{base_filename}_consensus_results.json"
    # Convert tuples to lists for JSON serialization
    json_data = {}
    for patient_id, disease_info in high_agreement_data.items():
        json_data[patient_id] = {
            'orpha_codes': disease_info['orpha_codes'],
            'disease_entities': disease_info['disease_entities'],
            'pairings': [list(pairing) for pairing in disease_info['pairings']]
        }
    
    with open(consensus_filename, 'w') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)
    print(f"Saved consensus results to: {consensus_filename}")
    
    return irr_filename, consensus_filename

# Example usage:
# Structure your data like this:
annotator_data = {
    'jeffrey': {
        'part1': jeffrey,  # your jeffrey variable
        'part2': jeffrey_p2  # your jeffrey_p2 variable
    },
    'jennifer': {
        'part1': jennifer,
        'part2': jennifer_p2
    },
    'lauren': {
        'part1': lauren,
        'part2': lauren_p2
    },
    'mya': {
        'part1': mya,
        'part2': mya_p2
    }
}

# Run comprehensive analysis
irr_results, high_agreement_rare_diseases = print_comprehensive_analysis(annotator_data, agreement_threshold=3)

# Save results
save_comprehensive_results(irr_results, high_agreement_rare_diseases)

# Access specific metrics:
mean_kappa = irr_results['agreement_metrics']['mean_pairwise_kappa']
fleiss_kappa = irr_results['agreement_metrics']['fleiss_kappa'] 
consensus_prevalence = len(high_agreement_rare_diseases) / irr_results['patient_level_metrics']['total_patients'] * 100

print(f"Key Results:")
print(f"- Mean Cohen's κ: {mean_kappa:.3f}")
print(f"- Fleiss' κ: {fleiss_kappa:.3f}")
print(f"- Consensus rare disease prevalence: {consensus_prevalence:.1f}%")

🔬 COMPREHENSIVE RARE DISEASE ANNOTATION ANALYSIS
Computing Inter-Rater Reliability Metrics...
Annotators: ['jeffrey', 'jennifer', 'lauren', 'mya']
Total patients: 223
jeffrey:
  - Patients with rare diseases: 188/223 (84.3%)
  - Total rare disease annotations: 255
jennifer:
  - Patients with rare diseases: 181/223 (81.2%)
  - Total rare disease annotations: 241
lauren:
  - Patients with rare diseases: 170/223 (76.2%)
  - Total rare disease annotations: 225
mya:
  - Patients with rare diseases: 183/223 (82.1%)
  - Total rare disease annotations: 247

Pairwise Cohen's Kappa (Patient-level rare disease presence):
--------------------------------------------------
jeffrey vs jennifer: κ = 0.765
jeffrey vs lauren: κ = 0.720
jeffrey vs mya: κ = 0.696
jennifer vs lauren: κ = 0.800
jennifer vs mya: κ = 0.671
lauren vs mya: κ = 0.608

Mean pairwise κ = 0.710

Overall Agreement Statistics:
- Perfect agreement (all annotators): 160/223 (71.7%)
- No rare diseases found by any annotator: 26/223 (11

In [17]:
def find_patients_without_phenotypes(high_agreement_data, phenotypes_per_patient):
    """
    Find all patient IDs that don't have matching phenotypes.
    
    Args:
        high_agreement_data: Dictionary from find_high_agreement_rare_diseases()
                            Structure: patient_id -> {orpha_codes: list, disease_entities: list, pairings: list}
        phenotypes_per_patient: Dictionary with patient_id -> phenotype data
                               Structure: patient_id -> {matched_phenotypes: list, original_text: str, stats: dict}
    
    Returns:
        List of patient IDs that don't have matching phenotypes
    """
    patients_without_phenotypes = []
    
    print(f"Checking {len(high_agreement_data)} patients for missing phenotypes...")
    print("-" * 60)
    
    for patient_id in high_agreement_data.keys():
        if patient_id not in phenotypes_per_patient:
            patients_without_phenotypes.append(patient_id)
            print(f"Missing: {patient_id}")
        else:
            # Check if phenotypes list is empty
            phenotypes_list = phenotypes_per_patient[patient_id]['matched_phenotypes']
            if not phenotypes_list:  # Empty list
                print(f"Empty phenotypes: {patient_id}")
            else:
                print(f"Has phenotypes: {patient_id} ({len(phenotypes_list)} phenotypes)")
    
    print("-" * 60)
    print(f"Found {len(patients_without_phenotypes)} patients without matching phenotypes")
    
    return patients_without_phenotypes


def save_patients_without_phenotypes(patients_without_phenotypes, filename=None):
    """
    Save list of patient IDs without phenotypes to a text file.
    
    Args:
        patients_without_phenotypes: List of patient IDs
        filename: Output filename. If None, uses timestamp
    
    Returns:
        String with filename
    """
    from datetime import datetime
    
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"patients_without_phenotypes_{timestamp}.txt"
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("PATIENTS WITHOUT MATCHING PHENOTYPES\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Total patients without phenotypes: {len(patients_without_phenotypes)}\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("Patient IDs:\n")
        f.write("-" * 20 + "\n")
        
        for patient_id in sorted(patients_without_phenotypes):
            f.write(f"{patient_id}\n")
    
    print(f"Saved {len(patients_without_phenotypes)} patient IDs to: {filename}")
    return filename


def find_patients_with_empty_phenotypes(phenotypes_per_patient):
    """
    Find patients that exist in phenotypes data but have empty matched_phenotypes lists.
    
    Args:
        phenotypes_per_patient: Dictionary with patient_id -> phenotype data
    
    Returns:
        List of patient IDs with empty phenotypes
    """
    patients_with_empty_phenotypes = []
    
    for patient_id, data in phenotypes_per_patient.items():
        if not data['matched_phenotypes']:  # Empty list
            patients_with_empty_phenotypes.append(patient_id)
    
    return patients_with_empty_phenotypes


def comprehensive_phenotype_analysis(high_agreement_data, phenotypes_per_patient, save_files=True):
    """
    Comprehensive analysis of phenotype coverage.
    
    Args:
        high_agreement_data: Dictionary from find_high_agreement_rare_diseases()
        phenotypes_per_patient: Dictionary with patient_id -> phenotype data
        save_files: Whether to save results to files
    
    Returns:
        Dictionary with analysis results
    """
    from datetime import datetime
    
    print("COMPREHENSIVE PHENOTYPE ANALYSIS")
    print("=" * 60)
    
    # Find patients without phenotype data
    patients_without_phenotypes = find_patients_without_phenotypes(high_agreement_data, phenotypes_per_patient)
    
    # Find patients with empty phenotypes
    patients_with_empty_phenotypes = find_patients_with_empty_phenotypes(phenotypes_per_patient)
    
    # Find patients in high agreement that have empty phenotypes
    high_agreement_patients_with_empty = []
    for patient_id in high_agreement_data.keys():
        if patient_id in phenotypes_per_patient:
            if not phenotypes_per_patient[patient_id]['matched_phenotypes']:
                high_agreement_patients_with_empty.append(patient_id)
    
    # Summary statistics
    total_high_agreement = len(high_agreement_data)
    total_with_phenotype_data = len([p for p in high_agreement_data.keys() if p in phenotypes_per_patient])
    total_with_actual_phenotypes = len([p for p in high_agreement_data.keys() 
                                      if p in phenotypes_per_patient and phenotypes_per_patient[p]['matched_phenotypes']])
    
    results = {
        'patients_without_phenotypes': patients_without_phenotypes,
        'patients_with_empty_phenotypes': high_agreement_patients_with_empty,
        'stats': {
            'total_high_agreement_patients': total_high_agreement,
            'patients_with_phenotype_data': total_with_phenotype_data,
            'patients_with_actual_phenotypes': total_with_actual_phenotypes,
            'patients_missing_phenotype_data': len(patients_without_phenotypes),
            'patients_with_empty_phenotypes': len(high_agreement_patients_with_empty)
        }
    }
    
    print("\nSUMMARY STATISTICS:")
    print("-" * 30)
    print(f"Total high agreement patients: {results['stats']['total_high_agreement_patients']}")
    print(f"Patients with phenotype data: {results['stats']['patients_with_phenotype_data']}")
    print(f"Patients with actual phenotypes: {results['stats']['patients_with_actual_phenotypes']}")
    print(f"Patients missing phenotype data: {results['stats']['patients_missing_phenotype_data']}")
    print(f"Patients with empty phenotypes: {results['stats']['patients_with_empty_phenotypes']}")
    
    if save_files:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Save patients without phenotypes
        if patients_without_phenotypes:
            save_patients_without_phenotypes(patients_without_phenotypes, 
                                           f"patients_without_phenotypes_{timestamp}.txt")
        
        # Save patients with empty phenotypes
        if high_agreement_patients_with_empty:
            filename = f"patients_with_empty_phenotypes_{timestamp}.txt"
            with open(filename, 'w', encoding='utf-8') as f:
                f.write("PATIENTS WITH EMPTY PHENOTYPES\n")
                f.write("=" * 50 + "\n\n")
                f.write(f"Total patients with empty phenotypes: {len(high_agreement_patients_with_empty)}\n")
                f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write("Patient IDs:\n")
                f.write("-" * 20 + "\n")
                
                for patient_id in sorted(high_agreement_patients_with_empty):
                    f.write(f"{patient_id}\n")
            
            print(f"Saved {len(high_agreement_patients_with_empty)} patient IDs with empty phenotypes to: {filename}")
        
        # Save comprehensive report
        report_filename = f"phenotype_analysis_report_{timestamp}.txt"
        with open(report_filename, 'w', encoding='utf-8') as f:
            f.write("COMPREHENSIVE PHENOTYPE ANALYSIS REPORT\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write("SUMMARY STATISTICS:\n")
            f.write("-" * 30 + "\n")
            for key, value in results['stats'].items():
                f.write(f"{key.replace('_', ' ').title()}: {value}\n")
            
            f.write(f"\nPATIENTS WITHOUT PHENOTYPE DATA ({len(patients_without_phenotypes)}):\n")
            f.write("-" * 50 + "\n")
            for patient_id in sorted(patients_without_phenotypes):
                f.write(f"{patient_id}\n")
            
            f.write(f"\nPATIENTS WITH EMPTY PHENOTYPES ({len(high_agreement_patients_with_empty)}):\n")
            f.write("-" * 50 + "\n")
            for patient_id in sorted(high_agreement_patients_with_empty):
                f.write(f"{patient_id}\n")
        
        print(f"Saved comprehensive report to: {report_filename}")
    
    return results


# Usage examples:

# Simple approach - just find and save patients without phenotypes:
patients_without_phenotypes = find_patients_without_phenotypes(high_agreement_rare_diseases, phenotypes_per_patient)
save_patients_without_phenotypes(patients_without_phenotypes, "missing_phenotypes_patients.txt")

# Comprehensive approach - full analysis with multiple files:
# analysis_results = comprehensive_phenotype_analysis(high_agreement_rare_diseases, phenotypes_per_patient)

# Access the results:
# missing_patients = analysis_results['patients_without_phenotypes']
# empty_patients = analysis_results['patients_with_empty_phenotypes']
# stats = analysis_results['stats']

Checking 178 patients for missing phenotypes...
------------------------------------------------------------
Has phenotypes: 10877494 (71 phenotypes)
Has phenotypes: 10402135 (147 phenotypes)
Has phenotypes: 10844136 (249 phenotypes)
Has phenotypes: 10914484 (59 phenotypes)
Has phenotypes: 10948957 (156 phenotypes)
Has phenotypes: 10981539 (88 phenotypes)
Has phenotypes: 10982872 (87 phenotypes)
Has phenotypes: 11208462 (151 phenotypes)
Has phenotypes: 11369570 (357 phenotypes)
Has phenotypes: 11561996 (210 phenotypes)
Has phenotypes: 11960419 (399 phenotypes)
Has phenotypes: 12125300 (148 phenotypes)
Has phenotypes: 12142402 (123 phenotypes)
Has phenotypes: 13387262 (214 phenotypes)
Has phenotypes: 13470702 (258 phenotypes)
Has phenotypes: 13496595 (220 phenotypes)
Has phenotypes: 13719735 (396 phenotypes)
Has phenotypes: 13993571 (378 phenotypes)
Has phenotypes: 14021673 (146 phenotypes)
Has phenotypes: 14186401 (167 phenotypes)
Has phenotypes: 14283383 (165 phenotypes)
Has phenotype

'missing_phenotypes_patients.txt'

In [19]:
import json
from collections import Counter

def parse_dataset_statistics(json_file_path):
    """
    Parse a JSON file containing patient data and extract dataset statistics.
    
    Args:
        json_file_path (str): Path to the JSON file
        
    Returns:
        dict: Dictionary containing dataset statistics
    """
    
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Initialize counters and lists
    num_patients = len(data)
    all_diseases = []
    phenotype_counts = []
    rare_disease_counts = []
    
    for patient_id, patient_data in data.items():
        # Extract unique diseases (from orpha_codes)
        orpha_codes = patient_data.get('orpha_codes', [])
        disease_entities = patient_data.get('disease_entities', [])
        
        # Count rare diseases per patient (using orpha_codes as proxy for rare diseases)
        rare_disease_counts.append(len(orpha_codes))
        
        # Collect all disease entities for unique count
        all_diseases.extend(disease_entities)
        
        # Count phenotypes per patient
        matched_phenotypes = patient_data.get('matched_phenotypes', [])
        phenotype_counts.append(len(matched_phenotypes))
    
    # Calculate statistics
    unique_diseases = len(set(all_diseases))
    avg_phenotypes_per_patient = sum(phenotype_counts) / num_patients if num_patients > 0 else 0
    avg_rare_diseases_per_patient = sum(rare_disease_counts) / num_patients if num_patients > 0 else 0
    
    # Create results dictionary
    statistics = {
        'num_patients': num_patients,
        'num_unique_diseases': unique_diseases,
        'avg_phenotypes_per_patient': round(avg_phenotypes_per_patient, 2),
        'avg_rare_diseases_per_patient': round(avg_rare_diseases_per_patient, 2)
    }
    
    return statistics

def print_latex_table_values(stats):
    """
    Print the statistics in a format ready for LaTeX table substitution.
    
    Args:
        stats (dict): Statistics dictionary from parse_dataset_statistics
    """
    print("LaTeX Table Values:")
    print(f"# of Patients: {stats['num_patients']}")
    print(f"# of Unique Diseases: {stats['num_unique_diseases']}")
    print(f"Avg. # of Phenotypes Per Patient: {stats['avg_phenotypes_per_patient']}")
    print(f"Avg. # of Rare Diseases Per Patient: {stats['avg_rare_diseases_per_patient']}")

# Example usage:
if __name__ == "__main__":
    # Replace 'your_file.json' with the actual path to your JSON file
    file_path = 'data/medical_students_data/high_agreement_with_phenotypes.json'
    
    try:
        stats = parse_dataset_statistics(file_path)
        print_latex_table_values(stats)
        
        # Also return the raw statistics for programmatic use
        print("\nRaw statistics dictionary:")
        print(stats)
        
    except FileNotFoundError:
        print(f"File {file_path} not found. Please check the file path.")
    except json.JSONDecodeError:
        print("Error decoding JSON file. Please check the file format.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Additional helper function to get detailed disease analysis
def get_detailed_disease_analysis(json_file_path):
    """
    Get more detailed analysis of the diseases in the dataset.
    
    Args:
        json_file_path (str): Path to the JSON file
        
    Returns:
        dict: Detailed disease analysis
    """
    
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    disease_frequency = Counter()
    orpha_code_frequency = Counter()
    
    for patient_data in data.values():
        disease_entities = patient_data.get('disease_entities', [])
        orpha_codes = patient_data.get('orpha_codes', [])
        
        # Count frequency of each disease
        for disease in disease_entities:
            disease_frequency[disease] += 1
            
        # Count frequency of each Orpha code
        for code in orpha_codes:
            orpha_code_frequency[code] += 1
    
    return {
        'most_common_diseases': disease_frequency.most_common(10),
        'most_common_orpha_codes': orpha_code_frequency.most_common(10),
        'total_unique_diseases': len(disease_frequency),
        'total_unique_orpha_codes': len(orpha_code_frequency)
    }

LaTeX Table Values:
# of Patients: 160
# of Unique Diseases: 123
Avg. # of Phenotypes Per Patient: 46.68
Avg. # of Rare Diseases Per Patient: 1.29

Raw statistics dictionary:
{'num_patients': 160, 'num_unique_diseases': 123, 'avg_phenotypes_per_patient': 46.68, 'avg_rare_diseases_per_patient': 1.29}
